In [23]:
!pip install -q transformers datasets peft accelerate bitsandbytes

without fine tunning

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType

In [25]:
model_name = "tiiuae/falcon-rw-1b"

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [27]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.bfloat16)

In [35]:
tokenizer.pad_token = tokenizer.eos_token  # Add padding token using eos_token

In [28]:
# Inference helper
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [29]:
# Test prompt
print(generate_response("Explain the theory of general relativity in simple terms."))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Explain the theory of general relativity in simple terms.
The theory of general relativity is a theory of the universe. It is a theory of the universe that explains the way the universe works. It is a theory that explains the way the universe works. It is a theory that explains the way the universe works. It is a theory that explains the way the universe works. It is a theory that explains the way the universe works. It is a theory that explains the way the universe works. It is a theory that explains the way the universe works.


With Parametric fine tunning

In [30]:
import json
from datasets import Dataset

In [31]:
# Load and format training data
with open("training_data.json", "r") as f:
    data = json.load(f)

In [32]:
# Convert to HF Dataset format
formatted_data = [{"text": f"### Instruction:\n{item['instruction']}\n### Response:\n{item['output']}"} for item in data]
dataset = Dataset.from_list(formatted_data)
dataset = dataset.train_test_split(test_size=0.1)

In [33]:
# Tokenize
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

In [36]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [37]:
#using LORA
from peft import get_peft_model, LoraConfig, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

In [38]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.1,
    bias="none"
)

In [39]:
# Add LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,313,198,080 || trainable%: 0.1198


In [41]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./falcon-peft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_dir="./logs",
    save_total_limit=1,
    save_strategy="epoch",

)

In [42]:
# Trainer
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


/tmp/ipython-input-42-108069179.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [43]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: btirkey1208 (btirkey1208-delhi-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=18, training_loss=1.722807036505805, metrics={'train_runtime': 210.9322, 'train_samples_per_second': 0.583, 'train_steps_per_second': 0.085, 'total_flos': 457272069193728.0, 'train_loss': 1.722807036505805, 'epoch': 3.0})

In [44]:
# Reload the fine-tuned model with LoRA weights
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): FalconForCausalLM(
      (transformer): FalconModel(
        (word_embeddings): Embedding(50304, 2048)
        (h): ModuleList(
          (0-23): 24 x FalconDecoderLayer(
            (self_attention): FalconAttention(
              (query_key_value): lora.Linear(
                (base_layer): FalconLinear(in_features=2048, out_features=6144, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=6144, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
            

In [45]:
# Test prompt
print(generate_response("What is the Fibonacci sequence and how is it generated?"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


What is the Fibonacci sequence and how is it generated?
The Fibonacci sequence is a mathematical sequence that is used to describe the growth of a series of numbers. The sequence is named after the Italian mathematician Leonardo Fibonacci.
The Fibonacci sequence is a sequence of numbers that is used to describe the growth of a series of numbers. The sequence is named after the Italian mathematician Leonardo Fibonacci.
The Fibonacci sequence is a sequence of numbers that is used to describe the growth of a series of numbers. The sequence is


In [ ]:

# Fix for tokenizer pad_token issue
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id


In [ ]:

# Split dataset into train and eval (90/10 split)
from datasets import DatasetDict

dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)


In [ ]:

from peft import LoraConfig, get_peft_model, TaskType

# PEFT Configuration
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none"
)

# Wrap model with PEFT
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:

from datasets import load_metric

# Simple exact match metric
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    exact_matches = [int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)]
    return {"exact_match": sum(exact_matches) / len(exact_matches)}


In [ ]:

# Test inference
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
result = pipe("Explain photosynthesis.", max_new_tokens=100)
print(result[0]['generated_text'])
